# Silver Date Dimension

Creates a reusable date dimension table in the silver layer for calendar dates from 2000-01-01 through 2050-12-31, including common reporting attributes and flags.

In [0]:
from pyspark.sql import functions as F

catalog_name = "michelin"
schema_name = "silver"
table_name = "dim_date"
full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
start_date = "2000-01-01"
end_date = "2050-12-31"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

base_dates = (
    spark.sql(
        f"""
        SELECT explode(sequence(to_date('{start_date}'), to_date('{end_date}'), interval 1 day)) AS calendar_date
        """
    )
)

dim_date_df = (
    base_dates
    .withColumn("date_key", F.date_format("calendar_date", "yyyyMMdd").cast("int"))
    .withColumn("calendar_year", F.year("calendar_date"))
    .withColumn("calendar_quarter", F.quarter("calendar_date"))
    .withColumn("quarter_name", F.concat(F.lit("Q"), F.quarter("calendar_date")))
    .withColumn("calendar_month", F.month("calendar_date"))
    .withColumn("month_name", F.date_format("calendar_date", "MMMM"))
    .withColumn("month_short_name", F.date_format("calendar_date", "MMM"))
    .withColumn("month_year", F.date_format("calendar_date", "yyyy-MM"))
    .withColumn("year_month_key", F.date_format("calendar_date", "yyyyMM").cast("int"))
    .withColumn("calendar_week", F.weekofyear("calendar_date"))
    .withColumn("week_start_date", F.date_sub(F.next_day("calendar_date", "Mon"), 7))
    .withColumn("week_end_date", F.date_add(F.date_sub(F.next_day("calendar_date", "Mon"), 7), 6))
    .withColumn("day_of_month", F.dayofmonth("calendar_date"))
    .withColumn("day_of_year", F.dayofyear("calendar_date"))
    .withColumn("day_of_week_iso", (((F.dayofweek("calendar_date") + F.lit(5)) % 7) + F.lit(1)).cast("int"))
    .withColumn("day_of_week_name", F.date_format("calendar_date", "EEEE"))
    .withColumn("day_name_short", F.date_format("calendar_date", "E"))
    .withColumn("is_weekend", F.col("day_of_week_iso").isin(6, 7))
    .withColumn("is_month_start", F.dayofmonth("calendar_date") == 1)
    .withColumn("is_month_end", F.last_day("calendar_date") == F.col("calendar_date"))
    .withColumn("quarter_start_date", F.to_date(F.date_trunc("quarter", "calendar_date")))
    .withColumn("quarter_end_date", F.date_sub(F.add_months(F.to_date(F.date_trunc("quarter", "calendar_date")), 3), 1))
    .withColumn("is_quarter_start", F.col("calendar_date") == F.col("quarter_start_date"))
    .withColumn("is_quarter_end", F.col("calendar_date") == F.col("quarter_end_date"))
    .withColumn("year_start_date", F.to_date(F.make_date(F.year("calendar_date"), F.lit(1), F.lit(1))))
    .withColumn("year_end_date", F.to_date(F.make_date(F.year("calendar_date"), F.lit(12), F.lit(31))))
    .withColumn("is_year_start", F.col("calendar_date") == F.col("year_start_date"))
    .withColumn("is_year_end", F.col("calendar_date") == F.col("year_end_date"))
    .select(
        "date_key",
        "calendar_date",
        "calendar_year",
        "calendar_quarter",
        "quarter_name",
        "calendar_month",
        "month_name",
        "month_short_name",
        "month_year",
        "year_month_key",
        "calendar_week",
        "week_start_date",
        "week_end_date",
        "day_of_month",
        "day_of_year",
        "day_of_week_iso",
        "day_of_week_name",
        "day_name_short",
        "is_weekend",
        "is_month_start",
        "is_month_end",
        "quarter_start_date",
        "quarter_end_date",
        "is_quarter_start",
        "is_quarter_end",
        "year_start_date",
        "year_end_date",
        "is_year_start",
        "is_year_end",
    )
)

dim_date_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)

print(f"Created {full_table_name} with {dim_date_df.count():,} rows from {start_date} to {end_date}.")

In [0]:
preview_df = spark.table("michelin.silver.dim_date").orderBy("calendar_date")
display(preview_df.limit(5))
print(f"Total rows: {preview_df.count():,}")